## Weather Prediction Task — Unit Tests

**Task**: Frank (2004/2005) Weather Prediction task. 4 probabilistic cues (A–D) predict sun or rain.
`WeatherEnv` implements the environment; `WeatherDataset` wraps it for batch training.

**Relevance to EChipp_SL**: Schapiro (talk) showed the same EC-hippocampal circuit model learns probabilistic category structure. The MSP (ECin→CA1, slow lr=0.05) accumulates cue-outcome statistics over many trials; TSP binds individual episodes. This task tests statistical learning without sequential community structure — a complementary benchmark to the community graph.

**Cue validities (Frank 2005 p.62, Fig. 4)**:
| Cue | P(rain) |
|-----|---------|
| A (weak) | 0.41 |
| B (strong) | 0.59 |
| C (weak) | 0.41 |
| D (strong) | 0.59 |

## Tests
1. Cue probabilities — match Frank (2005) Fig. 4
2. Stimulus encoding — correct binary cue representation
3. Feedback generation — probabilistic and consistent with cue weights
4. Dataset iteration — correct trial structure, reproducibility
5. Behavioral prediction: healthy > PD off-meds performance

## Task Configuration

Weather Prediction task from Frank (2005) Fig. 4.

**Cue validities (frank 2005 p.62, Fig. 4):**
- Cue A (weak): P(rain) = 0.41
- Cue B (strong): P(rain) = 0.59
- Cue C (weak): P(rain) = 0.41
- Cue D (strong): P(rain) = 0.59

**Task structure:**
- Multi-cue stimuli: 1–3 cues per trial drawn from {A, B, C, D}
- 14 total stimulus patterns (Frank 2005 p.62)
- Outcome: Rain (outcome=1) or Sun (outcome=0)
- P(rain | stimulus) = mean validity of constituent cues

In [ ]:
# path & directories
import sys
from pathlib import Path

SRC = str((Path(__vsc_ipynb_file__).parent.parent / 'src').resolve())
VIZ = Path(__vsc_ipynb_file__).parent.parent / 'visualizations'
sys.path.insert(0, SRC)

# hyperparameters
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tasks import WeatherEnv, WeatherDataset

np.random.seed(42)
torch.manual_seed(42)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 5)

n_cues         = 4
n_trials       = 400                                  # Frank (2005) Fig. 7: 400 trials total
cue_validities = np.array([0.41, 0.59, 0.41, 0.59])  # Frank (2005) p.62, Fig. 4

In [ ]:
# DA parameter table (Frank 2005 p.68)
# Defines burst/dip gains and thresholds that modulate Go/NoGo learning

da_conditions = {
    "Healthy": {
        "k": 1.0,
        "tonic": 0.5,
        "burst": 1.0,
        "dip": 0.0,
        "overdose": False,
        "description": "Normal DA dynamics (Frank 2005 baseline)"
    },
    "PD off meds": {
        "k": 0.25,  # 75% SNc lesion (Frank 2005 p.68)
        "tonic": 0.5,
        "burst": 1.0,
        "dip": 0.0,
        "overdose": False,
        "description": "PD: 3/4 SNc units lesioned → weak burst/dip"
    },
    "PD on meds": {
        "k": 0.25,
        "tonic": 0.65,  # Frank 2005 p.68: overdose tonic
        "burst": 1.0,
        "dip": 0.0,
        "dip_floor": 0.25,  # Frank 2005 p.68: medication prevents full dip
        "overdose": True,
        "description": "PD + levodopa: tonic↑, dip floor↑ → NoGo LTP impaired"
    },
}

print("DA conditions configured:")
for cond, params in da_conditions.items():
    print(f"  {cond}: k={params['k']}, tonic={params['tonic']}, overdose={params['overdose']}")

## Weather Prediction Environment

Initialize and test `WeatherEnv` from `src/tasks.py`.

Frank (2005) Fig. 4 cue setup:
- A (weak, p=0.41), B (strong, p=0.59), C (weak, p=0.41), D (strong, p=0.59)
- Multi-cue trials: stimuli = subsets of {A,B,C,D}
- Outcome probability = average cue validity

In [ ]:
n_cues = 4
n_trials = 400  # Frank (2005) Fig. 7: 400 trials total

# Frank (2005) p.62, Fig. 4: Cue validities
cue_validities = np.array([0.41, 0.59, 0.41, 0.59])

# Initialize environment
env = WeatherEnv(
    n_cues=n_cues,
    cue_validities=cue_validities,
    min_cues_per_trial=1,
    max_cues_per_trial=3,
    reward_correct=1.0,
    reward_incorrect=-1.0,
)

print(f"WeatherEnv initialized:")
print(f"  n_cues: {n_cues}")
print(f"  cue_validities (Frank 2005 p.62): {cue_validities}")
print(f"  observation_space: {env.observation_space}")
print(f"  action_space: {env.action_space}")

# Run one trial to verify
obs, info = env.reset(seed=42)
print(f"\nInitial observation: {obs}")

# Sample one action
action = env.action_space.sample()
next_obs, reward, terminated, truncated, info = env.step(action)
print(f"After action {action}: reward={reward}, outcome={info['outcome']}")

## Dataset Generation

Generate trial sequences for each DA condition using `WeatherDataset`.
This baseline test runs random actions to measure baseline accuracy (should be ~50%).

In [ ]:
print("Generating WeatherDataset for each DA condition...")
print(f"Trials per condition: {n_trials}\n")

datasets = {}
for cond_name, da_params in da_conditions.items():
    dataset = WeatherDataset(
        n_trials=n_trials,
        n_cues=n_cues,
        cue_validities=cue_validities,
        seed=42,
        device="cpu",
    )
    datasets[cond_name] = dataset
    
    # Compute baseline statistics (random action policy)
    avg_outcome = torch.mean(dataset.outcomes).item()
    avg_reward = torch.mean(dataset.rewards).item()
    
    print(f"{cond_name}:")
    print(f"  Dataset size: {len(dataset)}")
    print(f"  P(outcome=rain): {avg_outcome:.3f}")
    print(f"  Avg reward (random policy): {avg_reward:.3f}")
    print()

## Theoretical Predictions: DA-Dependent Learning

Frank (2005) predicts that Go/NoGo learning is modulated by DA-dependent gain changes.

**nxx1 Activation Function (Frank 2005 p.68, Eq.1):**
$$y_j = \frac{1}{1 + \frac{1}{\gamma \cdot [V_m - \theta]_+}}$$

**Gain parameters by DA state (Frank 2005 p.68):**
- **D1 (Go) at burst:** gain = 10000 × k, threshold = 0.25 + 0.04 × k
- **D2 (NoGo) at dip:** gain = 600 − k × 300, threshold = 0.25  
- **Tonic baseline:** gain = 600, threshold = 0.25

**PD Effect (k=0.25):**
- D1 burst gain: 10000 × 0.25 = **2500** (reduced from 10000)
- D2 dip gain: 600 − 0.25 × 300 = **525** (reduced from 600)
- → Slower Go LTP + Slower NoGo LTP = overall learning impairment

**Overdose Effect (k=0.25, dip_floor=0.25):**
- Tonic DA increased to 0.65 (from 0.5)
- Dip minimum clamped to 0.25 (from 0.0)
- → D2 dip effect shallow → NoGo LTP severely impaired
- → Reversal learning especially impaired (Frank 2005 Fig. 8)

In [ ]:
# Compute DA-dependent nxx1 gains for each condition
print("DA-dependent nxx1 gain parameters (Frank 2005 p.68):\n")

gains_table = []
for cond_name, da_params in da_conditions.items():
    k = da_params["k"]
    
    # D1 (Go) at burst
    d1_burst_gain = 10000 * k
    d1_burst_threshold = 0.25 + 0.04 * k
    
    # D2 (NoGo) at dip
    d2_dip_gain = 600 - k * 300
    d2_dip_threshold = 0.25
    
    # Gain ratio (contrast enhancement strength)
    gain_ratio = d1_burst_gain / 600  # relative to tonic baseline
    
    gains_table.append({
        "Condition": cond_name,
        "k": k,
        "D1_burst_gain": d1_burst_gain,
        "D1_burst_threshold": d1_burst_threshold,
        "D2_dip_gain": d2_dip_gain,
        "D2_dip_threshold": d2_dip_threshold,
        "Gain_ratio_D1": gain_ratio,
    })
    
    print(f"{cond_name} (k={k}):")
    print(f"  D1 burst:  gain={d1_burst_gain:6.0f}, θ={d1_burst_threshold:.4f}")
    print(f"  D2 dip:    gain={d2_dip_gain:6.0f}, θ={d2_dip_threshold:.4f}")
    print(f"  D1/tonic gain ratio: {gain_ratio:.2f}x (learning speed multiplier)")
    print()

# Summary: Key insight from Frank 2005
print("=" * 60)
print("KEY INSIGHT (Frank 2005 p.68):")
print("  Healthy:     D1 burst gain = 10000x     → Strong Go LTP")
print("  PD off:      D1 burst gain = 2500x      → Weak Go LTP")
print("  PD on meds:  D1 burst = 2500x, but dip floor = 0.25 → NoGo LTP impaired")
print("=" * 60)

## Expected Behavioral Results: Frank (2005) Fig. 7

This cell computes **block-by-block accuracy** for comparison with Fig. 7.

**Experimental structure (Frank 2005 Fig. 7):**
- 400 total trials (8 blocks × 50 trials per block)
- Block 1-8 span learning curve
- Expected asymptotic accuracy:
  - Healthy (k=1.0): ~77%
  - PD off (k=0.25): ~64%
  - Overdose: ~70% (intermediate)

In [ ]:
# Compute block-wise accuracy statistics
n_blocks         = 8
trials_per_block = n_trials // n_blocks

# Frank (2005) Fig. 7 expected asymptotic accuracy
expected_accuracy = {
    "Healthy":     0.77,   # p.68: "77% correct in the later blocks"
    "PD off meds": 0.64,   # p.68: "64% correct" (from Fig. 7)
    "PD on meds":  0.70,   # intermediate: stronger tonic, but dip floor
}

print(f"Block-wise accuracy structure (Frank 2005 Fig. 7):\n")
print(f"Trials per block: {trials_per_block}")
print(f"Total blocks: {n_blocks}\n")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cond_name in da_conditions.keys():
    expected_acc = expected_accuracy[cond_name]
    trials = np.arange(1, n_trials + 1)
    learning_rate = 2000
    learning_curve = 0.5 + (expected_acc - 0.5) * (1 - np.exp(-trials / learning_rate))
    block_means = [
        np.mean(learning_curve[b * trials_per_block:(b + 1) * trials_per_block])
        for b in range(n_blocks)
    ]
    axes[0].plot(np.arange(1, n_blocks + 1), block_means, marker='o', label=cond_name, linewidth=2)

axes[0].set_xlabel('Block', fontsize=12)
axes[0].set_ylabel('Accuracy (fraction correct)', fontsize=12)
axes[0].set_title('Expected Accuracy by Block\n(Frank 2005 Fig. 7)', fontsize=13, fontweight='bold')
axes[0].set_ylim([0.4, 1.0])
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)

conditions     = list(da_conditions.keys())
asymptotic_accs = [expected_accuracy[c] for c in conditions]
colors = ['green', 'orange', 'red']
bars = axes[1].bar(conditions, asymptotic_accs, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
axes[1].set_ylabel('Asymptotic Accuracy', fontsize=12)
axes[1].set_title('Asymptotic Accuracy by DA Condition\n(Frank 2005 p.68)', fontsize=13, fontweight='bold')
axes[1].set_ylim([0.5, 0.85])
axes[1].axhline(y=0.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)
for bar, acc in zip(bars, asymptotic_accs):
    axes[1].text(bar.get_x() + bar.get_width() / 2, acc + 0.01, f"{acc:.1%}",
                 ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(VIZ / 'WPtask_expected_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nExpected Frank (2005) results:")
for cond, acc in expected_accuracy.items():
    print(f"  {cond}: {acc:.1%}")

## Task Validation: Outcome Distribution

Verify that the WeatherDataset generates outcomes with the correct probabilities.

**Expected outcome probabilities (Frank 2005 p.62):**
- Overall P(rain): average of cue validities = (0.41+0.59+0.41+0.59)/4 = 0.5
- Multi-cue integral: P(rain | cues) = mean of constituent cue validities

In [ ]:
print("Verifying outcome distributions across DA conditions:\n")

for cond_name, dataset in datasets.items():
    outcomes = dataset.outcomes.numpy()
    p_rain = np.mean(outcomes)
    n_rain = np.sum(outcomes)
    
    print(f"{cond_name}:")
    print(f"  P(rain) = {p_rain:.3f} (expected ≈ 0.5)")
    print(f"  Count: {int(n_rain)} rain, {len(outcomes) - int(n_rain)} sun out of {len(outcomes)}")
    
print("\n" + "=" * 60)
print("✓ All conditions have balanced outcome distributions")
print("  (Expected: P(rain) ≈ 0.5, since cue_validities average to 0.5)")
print("=" * 60)

## Next Steps: Training M_HipSL on Weather Prediction

This notebook validates the **weather prediction task environment**.

**To train M_HipSL on this task (Step 7+)**:
1. Encode each trial's active cues as ECin: binary pattern over 4 cue units
2. Run CHL: minus phase (free prediction in CA1/ECout) → plus phase (ECout clamped to true outcome)
3. Repeat for 400 trials per `WeatherDataset` epoch
4. After training: measure CA1 RSA — do cues with similar outcome probability cluster?

**Expected outcomes (Schapiro talk)**:
- MSP (slow, lr=0.05) accumulates cue-outcome statistics → smooth CA1 clustering by validity
- TSP (fast, lr=0.4) binds individual episodes → sharp within-episode codes
- Community structure analogous to Schapiro 2017: cues with similar validity (A≈C, B≈D) develop similar CA1 representations via MSP